# Analiza eksploracyjna surowych danych (EDA)

Wstępny przegląd surowych danych transakcyjnych hurtowni (2021–2025) przed przygotowaniem zbiorów pod modele. Notebook nie zapisuje żadnych plików, służy do rozpoznania struktury danych i problemów z jakością.

### 1. Wczytanie i połączenie danych
Plik transakcji (`Sprzedaz.csv`) łączony jest ze słownikiem produktów (`ProduktyKategorie.csv`) po kluczu `ID_artykułu` (`LEFT JOIN`). Zbiór zawiera 196 045 transakcji.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Wczytanie plików
df1 = pd.read_csv(r'../dane/surowe/Sprzedaz.csv',sep=';', encoding='utf-8-sig')  # sprzedaż
df2 = pd.read_csv(r'../dane/surowe/ProduktyKategorie.csv',sep=';', encoding='utf-8-sig')  # dane artykułów

# Łączenie po kolumnie ID_artykułu 
df_merged = pd.merge(df1, df2, on="ID_artykułu", how="left") 

sales = df_merged
sales.head()

,Klient,data_sprzedaży,ID_artykułu,Cena_zakupu,Cena_sprzedaży,Ilość,Wartość_netto,Stawka_VAT,Nazwa,grupa_towarów,producent,rodzaj_towarów,pierwsze_nazwa,rodzaj_towaru
0,S2,4.01.2021,529,"19,52","22,49","4,84","108,85",5%,Boczek wędzony extra <>,Zygmunt,Zygmunt,Wędliny,Boczki i Bekony,4. Wędzonki i Boczki
1,S2,4.01.2021,7376,"19,05","21,99","5,06","111,27",5%,Kiełbasa góralska,Zygmunt,Zygmunt,Wędliny,Kiełbasy tradycyjne i regionalne,1. Kiełbasy Cienkie
2,S2,4.01.2021,6,"17,8","19,99","2,12","42,38",5%,Kiełbasa szynkowa wp.extra,ALEKSANDRIA,Aleksandria,Wędliny,Szynkowe i Parzone,2. Kiełbasy Grube i Suche
3,S2,4.01.2021,6941,1,"1,29","5,5","7,1",5%,Porcja z kurczaka,DRÓB,Yxx,DRÓB,Drób,9. Mięso Świeże i Surowe
4,S2,4.01.2021,203,"14,8","16,35","0,92","15,04",5%,Senacka kiełbasa,H.A.M,Ham,H.A.M,Kiełbasy Codzienne i Grillowe,1. Kiełbasy Cienkie


### 2. Konwersja typów danych
Data sprzedaży konwertowana jest na `datetime`, a kolumny liczbowe zapisane z przecinkiem dziesiętnym (ceny, ilość, wartość netto) na `float64`. Kolumny z kategoryzacji otrzymują czytelne nazwy: `pierwsze_nazwa` → `Podkategoria`, `rodzaj_towaru` → `Kategoria`.

In [4]:
sales['data_sprzedaży'] = pd.to_datetime(sales['data_sprzedaży'],dayfirst=True, errors='coerce')

sales['ID_artykułu'] = sales['ID_artykułu'].astype('int64')
sales['Cena_zakupu'] = sales['Cena_zakupu'].str.replace(',', '.').astype('float64')
sales['Cena_sprzedaży'] = sales['Cena_sprzedaży'].str.replace(',', '.').astype('float64')
sales['Ilość'] = sales['Ilość'].str.replace(',', '.').astype('float64')
sales['Wartość_netto'] = sales['Wartość_netto'].str.replace(',', '.').astype('float64')

sales = sales.rename(columns={'pierwsze_nazwa': 'Podkategoria'})
sales = sales.rename(columns={'rodzaj_towaru': 'Kategoria'})


sales.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196045 entries, 0 to 196044
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Klient          196045 non-null  object        
 1   data_sprzedaży  196045 non-null  datetime64[ns]
 2   ID_artykułu     196045 non-null  int64         
 3   Cena_zakupu     196045 non-null  float64       
 4   Cena_sprzedaży  196045 non-null  float64       
 5   Ilość           196045 non-null  float64       
 6   Wartość_netto   196045 non-null  float64       
 7   Stawka_VAT      196045 non-null  object        
 8   Nazwa           196045 non-null  object        
 9   grupa_towarów   196045 non-null  object        
 10  producent       196045 non-null  object        
 11  rodzaj_towarów  196045 non-null  object        
 12  Podkategoria    196045 non-null  object        
 13  Kategoria       196045 non-null  object        
dtypes: datetime64[ns](1), float64(4), in

### 3. Usunięcie zbędnych kolumn
Usunięto kolumny opisowe, które nie będą wykorzystywane w analizie (stawka VAT, grupa i rodzaj towarów, nazwa produktu, producent). Zużycie pamięci spada z ok. 131 MB do 59 MB.

In [5]:
sales = sales.drop(columns=['Stawka_VAT','grupa_towarów','rodzaj_towarów', 'Nazwa','producent'])

In [6]:
sales.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196045 entries, 0 to 196044
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Klient          196045 non-null  object        
 1   data_sprzedaży  196045 non-null  datetime64[ns]
 2   ID_artykułu     196045 non-null  int64         
 3   Cena_zakupu     196045 non-null  float64       
 4   Cena_sprzedaży  196045 non-null  float64       
 5   Ilość           196045 non-null  float64       
 6   Wartość_netto   196045 non-null  float64       
 7   Podkategoria    196045 non-null  object        
 8   Kategoria       196045 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(3)
memory usage: 59.0 MB


### 4. Kontrola jakości danych
* **Duplikaty:** 467 całkowicie zduplikowanych wierszy (ten sam klient, data, artykuł, ceny i ilość).
* **Wartości ujemne:** 2 transakcje z ujemną ilością (zwroty / korekty).
* **Mikro-zamówienia:** transakcje poniżej 0,5 kg.

Wszystkie te przypadki są usuwane w notebookach przygotowujących dane (03 i 04).

In [7]:
sales.duplicated().sum()

np.int64(467)

In [9]:
sales[sales.duplicated(keep=False)]

,Klient,data_sprzedaży,ID_artykułu,Cena_zakupu,Cena_sprzedaży,Ilość,Wartość_netto,Podkategoria,Kategoria
323,S4,2021-01-07,3550,8.36,9.82,3.34,32.80,Galareta,8. Garmażeria i Dania Gotowe
364,S4,2021-01-07,3550,8.36,9.82,3.34,32.80,Galareta,8. Garmażeria i Dania Gotowe
475,S2,2021-01-08,176,12.99,16.80,1.24,20.83,Frankfurterki,5. Parówki i Serdelki
481,S2,2021-01-08,176,12.99,16.80,1.24,20.83,Frankfurterki,5. Parówki i Serdelki
1080,S1,2021-01-14,5825,14.70,16.69,0.50,8.35,Pasztetowe i Smarowne,7. Wyroby Podrobowe
...,...,...,...,...,...,...,...,...,...
195461,S2,2025-12-22,6941,1.50,1.99,2.08,4.14,Drób,9. Mięso Świeże i Surowe
195594,S2,2025-12-29,6,22.49,25.99,2.10,54.58,Szynkowe i Parzone,2. Kiełbasy Grube i Suche
195597,S2,2025-12-29,7012,14.00,16.79,1.78,29.89,Wieprzowina,9. Mięso Świeże i Surowe
195601,S2,2025-12-29,6,22.49,25.99,2.10,54.58,Szynkowe i Parzone,2. Kiełbasy Grube i Suche


In [10]:
sales[sales['Ilość'] <0]

,Klient,data_sprzedaży,ID_artykułu,Cena_zakupu,Cena_sprzedaży,Ilość,Wartość_netto,Podkategoria,Kategoria
60948,S4,2022-09-07,16959,12.0,14.04,-1.50,-21.06,Kaszanki i Krupnioki,7. Wyroby Podrobowe
83267,S2,2023-04-26,8777,21.9,25.99,-6.24,-162.18,Galareta,8. Garmażeria i Dania Gotowe


In [13]:
sales[sales['Ilość'] < 0.5]

,Klient,data_sprzedaży,ID_artykułu,Cena_zakupu,Cena_sprzedaży,Ilość,Wartość_netto,Podkategoria,Kategoria
973,S2,2021-01-13,486,36.35,40.99,0.48,19.68,Salami i Wędliny Dojrzewające,2. Kiełbasy Grube i Suche
1762,R4,2021-01-21,11224,25.49,28.99,0.45,13.05,Kabanos,6. Kabanosy i Przekąski
2148,S2,2021-01-26,8745,13.99,14.99,0.48,7.20,Drób,9. Mięso Świeże i Surowe
2960,S2,2021-02-04,14325,14.03,15.99,0.36,5.76,Pieczenie i Rolady,3. Szynki i Schaby
3187,S5,2021-02-06,16125,42.40,50.88,0.40,20.35,Salami i Wędliny Dojrzewające,2. Kiełbasy Grube i Suche
...,...,...,...,...,...,...,...,...,...
194058,R1,2025-12-12,4819,34.40,41.11,0.34,13.98,Schaby i Polędwice,3. Szynki i Schaby
194712,S4,2025-12-16,19513,40.50,22.00,0.34,7.48,Produkty Dojrzewające i Wołowe,3. Szynki i Schaby
195094,R1,2025-12-19,4366,38.69,46.62,0.44,20.51,Kabanos,6. Kabanosy i Przekąski
195700,S4,2025-12-29,4938,32.99,37.99,0.46,17.48,Kiełbasy Suche i Podsuszane,1. Kiełbasy Cienkie
